In [2]:
import requests
import re
from collections import Counter
from typing import List

class SimpleRAG:
    def __init__(self, model_name="llama3:8b", base_url="http://localhost:11434"):
        """
        初始化簡單RAG系統
        
        Args:
            model_name: Ollama模型名稱
            base_url: Ollama服務URL
        """
        self.model_name = model_name
        self.base_url = base_url
        self.chat_url = f"{base_url}/api/chat"
        self.documents = []  # 知識庫文檔
        self.processed_docs = []  # 預處理後的文檔（用於計算相似度）
    
    def _preprocess_text(self, text: str) -> List[str]:
        """
        改進的文本預處理：更好地處理中文分詞
        """
        # 保留中文字符、英文字母、數字
        text = re.sub(r'[^\w\s\u4e00-\u9fff]', ' ', text)
        
        # 中文按字符分割，英文按詞分割
        words = []
        current_word = ""
        
        for char in text:
            if '\u4e00' <= char <= '\u9fff':  # 中文字符
                if current_word:  # 如果有未完成的英文詞
                    words.append(current_word.lower())
                    current_word = ""
                words.append(char)  # 中文字符直接加入
            elif char.isalnum():  # 英文字母或數字
                current_word += char
            else:  # 空格或其他分隔符
                if current_word:
                    words.append(current_word.lower())
                    current_word = ""
        
        if current_word:  # 處理最後一個詞
            words.append(current_word.lower())
        
        # 過濾掉空字符
        return [word for word in words if word.strip()]
    
    def _calculate_tf_idf_similarity(self, query_words: List[str], doc_words: List[str]) -> float:
        """
        改進的TF-IDF相似度計算
        """
        if not query_words or not doc_words:
            return 0.0
        
        # 轉換為集合以提高查找效率
        doc_word_set = set(doc_words)
        doc_word_count = Counter(doc_words)
        
        # 計算匹配的詞數和權重
        total_score = 0.0
        matched_words = 0
        
        for query_word in query_words:
            if query_word in doc_word_set:
                matched_words += 1
                # TF: 詞在文檔中的頻率
                tf = doc_word_count[query_word] / len(doc_words)
                # 簡化權重：越長的詞權重越高
                word_weight = len(query_word) if len(query_word) > 1 else 1
                total_score += tf * word_weight
        
        if matched_words == 0:
            return 0.0
        
        # 正規化：考慮匹配詞的比例
        match_ratio = matched_words / len(query_words)
        return total_score * match_ratio
    
    def _jaccard_similarity(self, query_words: List[str], doc_words: List[str]) -> float:
        """
        計算Jaccard相似度
        """
        set1 = set(query_words)
        set2 = set(doc_words)
        
        intersection = set1 & set2
        union = set1 | set2
        
        return len(intersection) / len(union) if union else 0.0
    
    def add_documents(self, documents: List[str]):
        """
        添加文檔到知識庫
        """
        self.documents = documents
        self.processed_docs = [self._preprocess_text(doc) for doc in documents]
        print(f"✅ 已添加 {len(documents)} 個文檔到知識庫")
        
        # 顯示預處理結果用於除錯
        print("📝 文檔預處理示例:")
        for i, (doc, processed) in enumerate(zip(documents[:5], self.processed_docs[:5])):
            print(f"文檔{i+1}: {doc[:30]}...")
            print(f"處理後: {processed[:10]}...")
            
    def find_top_k_relevant_documents(self, query: str, method: str = "tfidf", k: int = 5) -> tuple[list[str], list[float]]:
        """
        找到前k個最相關的文檔，返回文檔內容列表和相似度分數列表    
        Args:
            query: 查詢文本
            method: 相似度計算方法 ("tfidf" 或 "jaccard")
            k: 回傳的前幾名相似文檔數量    
        Returns:
            (top_k_documents, top_k_similarities)
        """
        if not self.documents:  return [], []
    
        query_words = self._preprocess_text(query)
        print(f"🔍 查詢預處理: {query_words}")
    
        similarities = []
    
        for i, doc_words in enumerate(self.processed_docs):
            if method == "tfidf":
                similarity = self._calculate_tf_idf_similarity(query_words, doc_words)
            else:  # jaccard
                similarity = self._jaccard_similarity(query_words, doc_words)
            similarities.append((i, similarity))
    
        # 排序並取得前k名
        top_k = sorted(similarities, key=lambda x: x[1], reverse=True)[:k]
    
        print(f"\n📊 相似度前 {k} 高的文檔：")
        for rank, (doc_index, score) in enumerate(top_k, 1):
            print(f"🏅第{rank}名：文檔{doc_index + 1}，相似度 {score:.4f}")
    
        # 拆成兩個list：文件內容與相似度分數
        top_k_documents = [self.documents[i] for i, _ in top_k]
        top_k_similarities = [score for _, score in top_k]
    
        return top_k_documents, top_k_similarities
    
    def find_most_relevant_document(self, query: str, method: str = "tfidf") -> tuple:
        """
        找到最相關的文檔，返回文檔內容和相似度分數
        
        Args:
            query: 查詢文本
            method: 相似度計算方法 ("tfidf" 或 "jaccard")
        
        Returns:
            (best_document, best_similarity)
        """
        if not self.documents:
            return "", 0.0
        
        query_words = self._preprocess_text(query)
        print(f"🔍 查詢預處理: {query_words}")
        
        similarities = []
        
        for i, doc_words in enumerate(self.processed_docs):
            if method == "tfidf":
                similarity = self._calculate_tf_idf_similarity(query_words, doc_words)
            else:  # jaccard
                similarity = self._jaccard_similarity(query_words, doc_words)
            similarities.append((i, similarity))
            #print(f"文檔{i+1}相似度: {similarity:.4f}")
            
        # 🔽 在迴圈之外，排序並輸出前5名
        print("\n📊 相似度前五高的文檔：")
        top_5 = sorted(similarities, key=lambda x: x[1], reverse=True)[:5]
        for rank, (doc_index, score) in enumerate(top_5, 1):
            print(f"🏅第{rank}名：文檔{doc_index+1}，相似度 {score:.4f}")
            
        # 找到最高相似度的文檔
        best_doc_idx, best_similarity = max(similarities, key=lambda x: x[1])
        
        print(f"🎯 最佳匹配: 文檔{best_doc_idx + 1} (相似度: {best_similarity:.4f})")
        
        return self.documents[best_doc_idx], best_similarity
  
    def rag_top_k_query(self, query: str, method: str = "tfidf", k: int = 5, similarity_threshold: float = 0.01) -> str:
        """
        使用前K個最相關文檔進行RAG查詢（只納入超過相似度門檻的文檔）
    
        Args:
            query: 用戶查詢
            method: 檢索方法 ("tfidf" 或 "jaccard")
            k: 考慮的文檔數量（最多）
            similarity_threshold: 相似度閾值（超過此值的文檔才納入參考資料）
    
        Returns:
            回應內容（字串）
        """
        # 取得前K個相關文檔與相似度
        top_k_docs, top_k_similarities = self.find_top_k_relevant_documents(query, method, k=k)
    
        # 篩選出相似度超過門檻的文檔
        filtered_docs_scores = [
            (doc, score) for doc, score in zip(top_k_docs, top_k_similarities)
            if score > similarity_threshold
        ]
    
        if filtered_docs_scores:
            # 有文檔通過門檻 → 使用RAG模式
            filtered_docs = [doc for doc, _ in filtered_docs_scores]
            filtered_scores = [score for _, score in filtered_docs_scores]
    
            references = "\n\n---\n\n".join(filtered_docs)
            prompt = f"""請基於以下多筆參考資料回答問題：
    
    參考資料：
    {references}
    
    問題：請以繁體中文回答，{query}
    
    請根據參考資料回答問題。如果參考資料不完全涵蓋問題，可以結合你的知識進行補充說明。"""
    
            context_info = f"[使用RAG模式，納入 {len(filtered_docs)} 筆文檔，最高相似度: {max(filtered_scores):.4f}]"
        else:
            # 沒有任何文檔通過門檻 → 使用一般LLM回答模式
            prompt = f"""問題：請以繁體中文回答，{query}
    
    注意：我在知識庫中沒有找到與此問題直接相關的資料，請基於你的一般知識回答這個問題。"""
    
            context_info = f"[一般回答模式，前{k}文件中最高相似度: {max(top_k_similarities):.4f}]"
    
        print(f"📋 {context_info}")
    
        # 呼叫 LLM API
        try:
            response = requests.post(
                self.chat_url,
                json={
                    "model": self.model_name,
                    "messages": [{"role": "user", "content": prompt}],
                    "stream": False
                },
                headers={"Content-Type": "application/json"},
                timeout=600
            )
    
            if response.status_code == 200:
                result = response.json()
                return result["message"]["content"]
            else:
                return f"錯誤: HTTP {response.status_code} - {response.text}"
    
        except Exception as e:
            return f"請求錯誤: {str(e)}"
    
    def rag_query(self, query: str, method: str = "tfidf", similarity_threshold: float = 0.01) -> str:
        """
        RAG查詢：總是調用LLM，但會根據檢索結果調整提示詞
        
        Args:
            query: 用戶查詢
            method: 檢索方法 ("tfidf" 或 "jaccard")
            similarity_threshold: 相似度閾值，超過此值才認為找到相關文檔
        """
        # 檢索相關文檔
        best_doc, best_similarity = self.find_most_relevant_document(query, method)
        
        # 根據相似度決定提示詞策略
        if best_similarity > similarity_threshold:
            # 找到相關文檔，使用RAG模式
            prompt = f"""請基於以下參考資料回答問題：

參考資料：
{best_doc}

問題：請以繁體中文回答，{query}

請根據參考資料回答問題。如果參考資料不完全涵蓋問題，請回答**我不知道**"""
            
            context_info = f"[使用RAG模式，相似度: {best_similarity:.4f}]"
        else:
            # 沒有找到相關文檔，使用一般模式但告知LLM
            prompt = f"""問題：請以繁體中文回答，{query}

注意：我在知識庫中沒有找到與此問題直接相關的資料，請基於你的一般知識回答這個問題。"""
            
            context_info = f"[一般回答模式，最高相似度: {best_similarity:.4f}]"
        
        print(f"📋 {context_info}")
        
        # 總是調用LLM
        try:
            response = requests.post(
                self.chat_url,
                json={
                    "model": self.model_name,
                    "messages": [{"role": "user", "content": prompt}],
                    "stream": False
                },
                headers={"Content-Type": "application/json"},
                timeout=600
            )
            
            if response.status_code == 200:
                result = response.json()
                return result["message"]["content"]
            else:
                return f"錯誤: HTTP {response.status_code} - {response.text}"
                
        except Exception as e:
            return f"請求錯誤: {str(e)}"

# 使用範例
def main(data_file="114_碩士班招生簡章_分頁.txt", model_name="llama3:8b"):
    # RAG系統
    print(f"🤖 Ollama {model_name} 簡單RAG系統")
    print("=" * 50)
    
    # 初始化RAG系統
    rag = SimpleRAG(model_name)
    
    # 更好的知識庫（關鍵詞更明確）
    with open("114_碩士班招生簡章_分頁.txt", "r", encoding="utf-8") as f:
        content = f.read()
        documents = content.split("=== 第")
        documents = documents[1:]
        documents = ["=== 第" + page.strip() for page in documents]
    print(f"共讀入 {len(documents)} 頁。")
    print(documents[0][:20])  # 印出第1頁的前20個字元做確認
    print(documents[1][:20])  # 印出第2頁的前20個字元做確認
    print(documents[-2][:20])  # 印出倒數第2頁的前20個字元做確認
    print(documents[-1][:20])  # 印出倒數第1頁的前20個字元做確認
    
    # 添加文檔到知識庫
    rag.add_documents(documents)
    
    # 測試查詢
    print("\n🔍 請輸入查詢問題（輸入 quit 結束）")
    print("=" * 60)
    
    while True:
        query = input("❓ 問題: ").strip()
    
        # 檢查是否結束
        if query.lower() == "quit":
            print("\n✅ 結束測試。")
            break
    
        # 執行 RAG 查詢
        #answer = rag.rag_query(query, method="tfidf")
        answer = rag.rag_top_k_query(query, method="tfidf", k=3)
        print(f"🤖 回答: {answer}")
        print("-" * 50)

if __name__ == "__main__":
    main(data_file="114_碩士班招生簡章_分頁.txt", model_name="gpt-oss:20b")

🤖 Ollama gpt-oss:20b 簡單RAG系統
共讀入 49 頁。
=== 第1 頁 ===
113 年 9
=== 第2 頁 ===
114 學 年
=== 第48 頁 ===
世新大學 1
=== 第49 頁 ===
世新大學 1
✅ 已添加 49 個文檔到知識庫
📝 文檔預處理示例:
文檔1: === 第1 頁 ===
113 年 9 月 19 日本校 ...
處理後: ['第', '1', '頁', '113', '年', '9', '月', '19', '日', '本']...
文檔2: === 第2 頁 ===
114 學 年 度 碩 士 班 招...
處理後: ['第', '2', '頁', '114', '學', '年', '度', '碩', '士', '班']...
文檔3: === 第3 頁 ===
報名流程圖 
 
1.列印出之報名...
處理後: ['第', '3', '頁', '報', '名', '流', '程', '圖', '1', '列']...
文檔4: === 第4 頁 ===
世新大學 114 學年度碩士班招生...
處理後: ['第', '4', '頁', '世', '新', '大', '學', '114', '學', '年']...
文檔5: === 第5 頁 ===
附錄一：入學大學同等學力認定標準（...
處理後: ['第', '5', '頁', '附', '錄', '一', '入', '學', '大', '學']...

🔍 請輸入查詢問題（輸入 quit 結束）


❓ 問題:  資訊管理學士碩士班 招生名額


🔍 查詢預處理: ['資', '訊', '管', '理', '學', '士', '碩', '士', '班', '招', '生', '名', '額']

📊 相似度前 5 高的文檔：
🏅第1名：文檔4，相似度 0.1672
🏅第2名：文檔19，相似度 0.1454
🏅第3名：文檔28，相似度 0.1366
🏅第4名：文檔45，相似度 0.1216
🏅第5名：文檔3，相似度 0.1209
📋 [使用RAG模式，納入 5 筆文檔，最高相似度: 0.1672]
🤖 回答: 資訊管理學系碩士班（一般生）之招生名額為 **9 名**。
--------------------------------------------------


❓ 問題:  資訊管理學系聯絡方式為何?


🔍 查詢預處理: ['資', '訊', '管', '理', '學', '系', '聯', '絡', '方', '式', '為', '何']

📊 相似度前 5 高的文檔：
🏅第1名：文檔4，相似度 0.1158
🏅第2名：文檔16，相似度 0.0837
🏅第3名：文檔19，相似度 0.0829
🏅第4名：文檔39，相似度 0.0800
🏅第5名：文檔21，相似度 0.0736
📋 [使用RAG模式，納入 5 筆文檔，最高相似度: 0.1158]
🤖 回答: 
--------------------------------------------------


...

🔍 查詢預處理: ['資', '管', '系', '報', '名', '資', '格', '是', '什', '麼']

📊 相似度前 5 高的文檔：
🏅第1名：文檔3，相似度 0.1025
🏅第2名：文檔4，相似度 0.0895
🏅第3名：文檔10，相似度 0.0866
🏅第4名：文檔43，相似度 0.0744
🏅第5名：文檔42，相似度 0.0740
📋 [使用RAG模式，納入 5 筆文檔，最高相似度: 0.1025]
🤖 回答: **資訊管理學系（資管系）報名資格說明**

| 資格條件 | 具體要求 |
|----------|---------|
| **學歷** | 必須持有「資訊科技、資訊管理或相近領域」的學士學位（含同等學力）。 |
| **修業年限／學分** | 修完至少三學年度，且符合系所規定之學分門檻（一般為 90‑120 個學分，具體數字請參照本校「報考資格」章節）。 |
| **同一系所組限制** | 本校在讀或已獲准保留入學資格之研究生成績，無法再次申請相同系所組的招生考試。 |
| **其他基礎條件** | 需完成網路報名填表、繳交報名費（全額 NT$1,400）以及郵寄正式文件；若符合低收入戶或畢業生優待，可申請減免/全免報名費。 |
| **特殊需求** | 如有肢體障礙等身心障礙，需要在網路填表時說明並附上相關證明，以利安排適當試場環境。 |

> **註：** 以上為資訊管理學系的一般報考資格概述，實際細項（如最低學分數、特殊職業背景加分等）仍以「114 學年度碩士班招生簡章」第七章「招生系所組、名額、報考資格」及後續公告為準。若對具體數字或例外情形有疑問，請至官方網站查詢最新說明或直接洽詢招生辦公室。
--------------------------------------------------


❓ 問題:  報名費多少錢? 本校生有優惠嗎?


🔍 查詢預處理: ['報', '名', '費', '多', '少', '錢', '本', '校', '生', '有', '優', '惠', '嗎']

📊 相似度前 5 高的文檔：
🏅第1名：文檔43，相似度 0.1093
🏅第2名：文檔3，相似度 0.1003
🏅第3名：文檔10，相似度 0.0866
🏅第4名：文檔7，相似度 0.0779
🏅第5名：文檔42，相似度 0.0662
📋 [使用RAG模式，納入 5 筆文檔，最高相似度: 0.1093]
🤖 回答: **報名費用說明（世新大學 114 學年度碩士班招生）**

| 報名類別 | 原始報名費 | 優惠後實際繳交金額 |
|---------|------------|--------------------|
| **一般考生** | NT$1,400 | NT$1,400（如無資格申請減免） |
| **畢、肄業校友（含應屆畢業生）** | － | **NT$700（50% 減免）** |
| **低收入戶** | － | **全額免除（0 元）** |
| **中低收入戶** | － | **60% 減免，實際繳納 NT$560** |

> *備註*：  
> 1. 優惠的申請者（畢業校友、低收入戶、中低收入戶）皆必須先行完成報名手續並**先支付全額報名費（NT$1,400）**，再依照對應附表（2‑1 或 2‑2）及「領款／轉帳匯款資料表」(附表 2‑3)提交減免/免除申請。  
> 2. 所有優惠的審核結果於 **114 年 2 月 17** 日起開放查詢。  
> 3. 若逾期或文件不齊全，將不予受理。

---

### 本校生（畢、肄業校友）是否有優惠？

是的。世新大學
--------------------------------------------------


❓ 問題:  quit



✅ 結束測試。
